# Proyecto de regresión logística

En este notebook voy a hacer el proyecto de regresión logística paso a paso.  
La idea es intentar entender un poco los datos, prepararlos y luego entrenar un modelo sencillo para predecir la variable objetivo.


## 1. Importar librerías

Primero importo las librerías que voy a usar.  
Si alguna no está instalada, en la siguiente celda se puede instalar.


In [32]:
# Si te da error con pandas o sklearn, ejecuta esta celda una sola vez
# Si ya lo tienes instalado, no hace falta

import sys
!{sys.executable} -m pip install pandas numpy scikit-learn matplotlib seaborn -q



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from pickle import dump


## 2. Cargar los datos

Aquí cargo el dataset desde la URL del ejercicio.  
Así no necesito descargar nada manualmente.


In [34]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/logistic-regression-project-tutorial/main/bank-marketing-campaign-data.csv"
total_data = pd.read_csv(url, sep=";")
total_data.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 3. Revisión rápida

Voy a mirar el tamaño, duplicados y nulos para ver si hay algo raro antes de seguir.


In [35]:
total_data.shape


(41188, 21)

In [36]:
total_data = total_data.drop_duplicates().reset_index(drop=True)
total_data.shape


(41176, 21)

In [37]:
total_data.isnull().sum()


age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

## 4. Pasar texto a números y escalar

Como la regresión logística necesita números, convierto las columnas de texto a valores numéricos.  
Después hago un escalado Min-Max para dejar todo en una escala parecida.


In [38]:
from sklearn.preprocessing import MinMaxScaler

total_data["job_n"] = pd.factorize(total_data["job"])[0]
total_data["marital_n"] = pd.factorize(total_data["marital"])[0]
total_data["education_n"] = pd.factorize(total_data["education"])[0]
total_data["default_n"] = pd.factorize(total_data["default"])[0]
total_data["housing_n"] = pd.factorize(total_data["housing"])[0]
total_data["loan_n"] = pd.factorize(total_data["loan"])[0]
total_data["contact_n"] = pd.factorize(total_data["contact"])[0]
total_data["month_n"] = pd.factorize(total_data["month"])[0]
total_data["day_of_week_n"] = pd.factorize(total_data["day_of_week"])[0]
total_data["poutcome_n"] = pd.factorize(total_data["poutcome"])[0]
total_data["y_n"] = pd.factorize(total_data["y"])[0]
num_variables = ["job_n", "marital_n", "education_n", "default_n", "housing_n", "loan_n", "contact_n", "month_n", "day_of_week_n", "poutcome_n",
                 "age", "duration", "campaign", "pdays", "previous", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed", "y_n"]

scaler = MinMaxScaler()
scal_features = scaler.fit_transform(total_data[num_variables])
total_data_scal = pd.DataFrame(scal_features, index = total_data.index, columns = num_variables)
total_data_scal.head()

,job_n,marital_n,education_n,default_n,housing_n,loan_n,contact_n,month_n,day_of_week_n,poutcome_n,...,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y_n
0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.053070,0.0,1.0,0.0,0.9375,0.698753,0.60251,0.957379,0.859735,0.0
1,0.090909,0.0,0.142857,0.5,0.0,0.0,0.0,0.0,0.0,0.0,...,0.030297,0.0,1.0,0.0,0.9375,0.698753,0.60251,0.957379,0.859735,0.0
2,0.090909,0.0,0.142857,0.0,0.5,0.0,0.0,0.0,0.0,0.0,...,0.045954,0.0,1.0,0.0,0.9375,0.698753,0.60251,0.957379,0.859735,0.0
3,0.181818,0.0,0.285714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.030704,0.0,1.0,0.0,0.9375,0.698753,0.60251,0.957379,0.859735,0.0
4,0.090909,0.0,0.142857,0.0,0.0,0.5,0.0,0.0,0.0,0.0,...,0.062424,0.0,1.0,0.0,0.9375,0.698753,0.60251,0.957379,0.859735,0.0


## 5. Selección de variables

Aquí separo X e y, hago train/test y luego me quedo con algunas variables usando SelectKBest.  
No es la única forma, pero para el ejercicio me sirve.


In [39]:
X = total_data_scal.drop("y_n", axis=1)
y = total_data_scal["y_n"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

selection_model = SelectKBest(score_func=chi2, k=5)
selection_model.fit(X_train, y_train)

selected_features = X_train.columns[selection_model.get_support()]
selected_features


Index(['poutcome_n', 'previous', 'emp.var.rate', 'euribor3m', 'nr.employed'], dtype='str')

In [40]:
X_train_sel = pd.DataFrame(selection_model.transform(X_train), columns=selected_features)
X_test_sel = pd.DataFrame(selection_model.transform(X_test), columns=selected_features)

X_train_sel.head()


,poutcome_n,previous,emp.var.rate,euribor3m,nr.employed
0,0.0,0.0,1.000000,0.980730,1.000000
1,0.0,0.0,0.333333,0.138291,0.512287
2,0.0,0.0,0.937500,0.956926,0.859735
3,0.0,0.0,0.937500,0.957379,0.859735
4,0.0,0.0,0.333333,0.175924,0.512287


## 6. Guardar datos limpios

Guardo una copia por si luego quiero reutilizar el train y el test ya preparados.


In [41]:
X_train_sel["y_n"] = list(y_train)
X_test_sel["y_n"] = list(y_test)

os.makedirs("data/processed", exist_ok=True)

X_train_sel.to_csv("data/processed/clean_train.csv", index=False)
X_test_sel.to_csv("data/processed/clean_test.csv", index=False)


In [42]:
train_data = pd.read_csv("data/processed/clean_train.csv")
test_data = pd.read_csv("data/processed/clean_test.csv")

train_data.head()


,poutcome_n,previous,emp.var.rate,euribor3m,nr.employed,y_n
0,0.0,0.0,1.000000,0.980730,1.000000,0.0
1,0.0,0.0,0.333333,0.138291,0.512287,0.0
2,0.0,0.0,0.937500,0.956926,0.859735,0.0
3,0.0,0.0,0.937500,0.957379,0.859735,0.0
4,0.0,0.0,0.333333,0.175924,0.512287,0.0


## 7. Modelo base

Ahora entreno una regresión logística básica para ver qué resultado da sin tocar mucho.


In [43]:
X_train = train_data.drop("y_n", axis=1)
y_train = train_data["y_n"]

X_test = test_data.drop("y_n", axis=1)
y_test = test_data["y_n"]

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred[:10]


array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [44]:
accuracy_score(y_test, y_pred)


0.8919378338999514

In [45]:
confusion_matrix(y_test, y_pred)


array([[3615,   30],
       [ 415,   58]])

In [46]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

         0.0       0.90      0.99      0.94      3645
         1.0       0.66      0.12      0.21       473

    accuracy                           0.89      4118
   macro avg       0.78      0.56      0.57      4118
weighted avg       0.87      0.89      0.86      4118



## 8. Pequeña optimización

Pruebo unos pocos valores para ver si mejora un poco el modelo.


In [47]:
grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"]
    },
    cv=3,
    scoring="accuracy"
)

grid.fit(X_train, y_train)
grid.best_params_


{'C': 0.1, 'solver': 'lbfgs'}

In [48]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

y_pred_opt = best_model.predict(X_test)

print(confusion_matrix(y_test, y_pred_opt))
print(classification_report(y_test, y_pred_opt))
print("Accuracy:", accuracy_score(y_test, y_pred_opt))


[[3628   17]
 [ 420   53]]
              precision    recall  f1-score   support

         0.0       0.90      1.00      0.94      3645
         1.0       0.76      0.11      0.20       473

    accuracy                           0.89      4118
   macro avg       0.83      0.55      0.57      4118
weighted avg       0.88      0.89      0.86      4118

Accuracy: 0.8938805245264692


## 9. Guardar el modelo

Por último guardo el modelo por si luego lo quiero usar otra vez sin entrenarlo de nuevo.


In [49]:
os.makedirs("models", exist_ok=True)
dump(best_model, open("models/logistic_regression_model.sav", "wb"))


## Conclusión

En este proyecto he cargado el dataset, lo he preparado, he convertido variables categóricas a numéricas, he seleccionado algunas variables y he entrenado una regresión logística.

No es un análisis súper avanzado, pero me sirve para entender mejor el proceso completo y ver cómo montar un modelo sencillo de clasificación.
